[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/07_nativeness/07_nativeness_lab.ipynb)


# 07 — Nativeness · Naturalness — AbNatiV · Ab-RoBERTa

> 본문 [`07_nativeness.md`](07_nativeness.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행합니다.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나옵니다. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백합니다.
>
> **실측 소요 —** Ab-RoBERTa 5후보 × VH+VL **70초** · AbNatiV 스코어링 **3.5~6.6초**(단 체크포인트 내려받기 **약 33분 / 2.6GB** → 기본 비활성)

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`07_nativeness`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `07_nativeness/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

이 셀이 끝나면 두 개의 경로가 준비됩니다 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "07_nativeness"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib torch transformers"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌립니다 — pip 만으로는 hmmscan 이 없습니다.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — Ab-RoBERTa pseudo-likelihood (본문 7.2)

항체 전용 언어모델 `mogam-ai/Ab-RoBERTa` 로 각 position 을 차례로 마스킹해 **실제 잔기의 로그확률**을 모아 평균합니다(masked pseudo-LL).
스코어링 스크립트는 이 챕터 폴더에 함께 실려 있습니다 — `score_abroberta_pseudolikelihood.py`.

```bash
python score_abroberta_pseudolikelihood.py --fasta variants.fasta --out scores.csv
```

후보 서열은 **앞 랩에서 내가 만든 것**을 먼저 씁니다(Ch.05 Sapiens · Ch.06 Humatch/AnthroAb). 없으면 `data/variants.fasta` 로 폴백합니다.
**실측 70초** (5후보 × VH+VL = 10 체인 스캔).

In [ ]:
import pandas as pd

# 후보 모으기 — 내 결과 우선
cands = {"parental": (VH, VL)}
def _pick(chapter, fname, keys, label):
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p); print(f"[내 결과 · {chapter}] {p}")
        return (f[keys[0]], f[keys[1]])
    return None

got = _pick("05_humanize_sapiens", "sapiens_humanized_noguard.fasta",
            ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens")
if got: cands["sapiens"] = got
got = _pick("06_cdr_safe_tools", "humatch_humanised.fasta",
            ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch")
if got: cands["humatch"] = got
got = _pick("06_cdr_safe_tools", "anthroab_best_score.fasta",
            ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab")
if got: cands["anthroab"] = got
got = _pick("06_cdr_safe_tools", "anthroab_masked_FRonly.fasta",
            ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked")
if got: cands["anthroabFRmasked"] = got

if len(cands) == 1:
    print("[레퍼런스] data/variants.fasta 의 5개 후보를 씁니다")
    v = read_fasta("data/variants.fasta")
    for name in ["parental", "sapiens", "humatch", "anthroab", "anthroabFRmasked"]:
        cands[name] = (v[f"{name}_VH"], v[f"{name}_VL"])

fa = write_fasta(MY / "variants.fasta",
                 {f"{n}_{c}": s for n, (h, l) in cands.items() for c, s in (("VH", h), ("VL", l))})
print("\n후보:", ", ".join(cands), "→", fa)

In [ ]:
from score_abroberta_pseudolikelihood import score_paired    # 이 챕터 폴더의 스크립트

t0 = time.time()
rows = []
try:
    for name, (h, l) in cands.items():
        res = score_paired(h, l)
        for ch in ("VH", "VL", "paired"):
            rows.append({"variant": name, "chain": ch,
                         "mean_logp": res[ch]["mean_logp"],
                         "perplexity": res[ch]["perplexity"],
                         "n_residues": res[ch]["n"]})
        print(f"{name:18s} paired mean_logp={res['paired']['mean_logp']:+.4f}  "
              f"ppl={res['paired']['perplexity']:.4f}")
    abr = pd.DataFrame(rows)
    abr.to_csv(MY / "abroberta_scores_summary.csv", index=False)
    print(f"\nAb-RoBERTa 스코어링: {time.time()-t0:.1f}초 → {MY/'abroberta_scores_summary.csv'}")
except Exception as e:
    print("Ab-RoBERTa 실행 실패:", type(e).__name__, str(e)[:200])
    print("→ 레퍼런스(data/abroberta_scores_summary.csv)로 이어갑니다.")
    abr = pd.read_csv("data/abroberta_scores_summary.csv")

## 2) 내 결과 확인 — naturalness ≠ humanness (본문 7.2.2)

`mean_log_prob` 는 **높을수록**(0 에 가까울수록) 자연스럽고, `perplexity = exp(-mean_logp)` 는 **낮을수록** 좋습니다.

여기서 본문이 짚은 반직관적 사실을 직접 봅니다 — **VH 만 보면 parental 이 가장 자연스럽습니다.**
사람다움(humanness)과 자연스러움(naturalness)은 **다른 축**입니다.

In [ ]:
pv = abr.pivot(index="variant", columns="chain", values="mean_logp").round(4)
pv["perplexity(paired)"] = abr[abr.chain == "paired"].set_index("variant")["perplexity"].round(4)
display(pv.sort_values("paired", ascending=False))

best_vh = abr[abr.chain == "VH"].sort_values("mean_logp", ascending=False).iloc[0]
print(f"VH 만 보면 가장 자연스러운 것: {best_vh['variant']} ({best_vh['mean_logp']:+.4f})")
print("→ 사람다움(Sapiens humanness·Humatch CNN·AbNatiV)이 올라간 후보가 "
      "naturalness 에서는 오히려 내려갈 수 있습니다. Ab-RoBERTa 는 주 humanness 지표로 쓰면 안 됩니다.")

In [ ]:
from humanization_viz import humanness_bars
import numpy as np

# exp(mean_logp) = 잔기당 pseudo-likelihood 기하평균(0~1) → 바 차트로 보기 좋은 형태
others = [v for v in abr.variant.unique() if v != "parental"]
pick = "sapiens" if "sapiens" in others else others[0]
g = abr.set_index(["variant", "chain"])["mean_logp"]
bars = [{"chain": ch, "parental": float(np.exp(g[("parental", ch)])),
         "humanized": float(np.exp(g[(pick, ch)]))} for ch in ("VH", "VL", "paired")]
humanness_bars(bars, f"Ab-RoBERTa naturalness — parental vs {pick} (exp(mean logP))",
               "07_naturalness.png", ylabel="per-residue pseudo-likelihood")
print("VH 막대를 보세요 — humanized 가 parental 보다 낮습니다(사람다워졌지만 덜 '자연스러워짐').")
from IPython.display import Image; Image("07_naturalness.png")

## 3) 직접 실행 — AbNatiV nativeness (본문 7.1) · **기본 비활성**

AbNatiV 는 "사람 잔기를 얼마나 썼나"(humanness)가 아니라 **"그 조합이 실제 사람 항체로서 얼마나 자연스러운가"**(nativeness)를 봅니다.

**스코어링 자체는 4서열에 3.5~6.6초로 끝납니다.** 무거운 건 **체크포인트**입니다 —
`abnativ init` 은 9개 ckpt(~6GB)를 **순차로** 받아 매우 느립니다(실측: 30분에 3개). 실제로 필요한 4개만 **병렬로** 받으면 **약 33분 / 2.6GB**(실측)입니다.

그래서 아래 셀은 **`RUN_ABNATIV = False` 가 기본**입니다. 그대로 두면 커밋된 `data/abnativ_summary_all_models.csv`(실제 실행 산출물)로 이어집니다.
직접 돌리려면 `True` 로 바꾸세요.

In [ ]:
RUN_ABNATIV = False        # ← True 로 바꾸면 체크포인트를 받아 실제로 스코어링합니다

abn_csv = None
if RUN_ABNATIV:
    try:
        _ensure("abnativ anarci")
        # abnativ init 은 9개 ckpt 를 순차 다운로드해 매우 느립니다 → 필요한 4개만 병렬로(실측 33분)
        home = pathlib.Path.home() / ".abnativ/models/pretrained_models"
        home.mkdir(parents=True, exist_ok=True)
        base = "https://zenodo.org/record/17295347/files"
        need = [m for m in ["vh_model", "vlambda_model", "vh2_model", "vl2_model"]
                if not (home / f"{m}.ckpt").exists()]
        if need:   # 한 셸 안에서 병렬 다운로드 후 wait (순차 abnativ init 보다 훨씬 빠름)
            par = " ".join(f'wget -c -q -O "{home}/{m}.ckpt" "{base}/{m}.ckpt?download=1" &'
                           for m in need)
            _run(f"({par} wait)")

        write_fasta(MY / "abnativ_vh.fa", {f"{n}_VH": h for n, (h, l) in cands.items()})
        write_fasta(MY / "abnativ_vl.fa", {f"{n}_VL": l for n, (h, l) in cands.items()})
        t0 = time.time()
        for nat, fa_in, oid in (("VH", "abnativ_vh.fa", "vh"), ("VLambda", "abnativ_vl.fa", "vl"),
                                ("VH2", "abnativ_vh.fa", "vh2"), ("VL2", "abnativ_vl.fa", "vl2")):
            _run(f'CUDA_VISIBLE_DEVICES="" abnativ score -nat {nat} -i "{MY/fa_in}" '
                 f'-odir "{MY/"abnativ"}" -oid {oid} -align -ncpu 4')
        print(f"AbNatiV 스코어링: {time.time()-t0:.1f}초 (실측 3.5~6.6초/모델)")
        abn_csv = sorted((MY / "abnativ").glob("*_seq_scores.csv"))
    except Exception as e:
        print("AbNatiV 실행 실패:", type(e).__name__, str(e)[:200])
        print("→ 레퍼런스로 이어갑니다.")
else:
    print("RUN_ABNATIV = False → 커밋된 레퍼런스(data/abnativ_summary_all_models.csv)로 진행합니다.")
    print("  · 스코어링은 빠르지만(3.5~6.6초) 체크포인트 다운로드가 약 33분 / 2.6GB 입니다(실측).")

## 4) 내 결과 / 레퍼런스 — nativeness 프로파일 (본문 7.1.3)

AbNatiV 에는 **두 세대**가 있고, 본문의 두 표는 사실 **서로 다른 모델**입니다 — 이걸 모르면 숫자가 모순돼 보입니다.

- **AbNatiV1**(`-nat VH` / `-nat VLambda`) — 본문 7.1.3 의 주 표(0.648 → 0.880 · FR 0.632 → 0.925 · CDR-H3 0.614 → 0.626 · VL 0.902). **7개 숫자 전부 재현**됩니다.
- **AbNatiV2**(`-nat VH2` / `-nat VL2`) — 본문 7.2.3 패널의 "AbNatiV H" 열. parental **0.4927 은 재현**되지만, humanized 의 **0.6900 은 재현되지 않습니다** — 실측은 **0.7777**.

그리고 본문의 "VL 0.902" 는 **parental** VL 입니다(humanized VL 은 0.9980). 이것도 표에서 확인합니다.

In [ ]:
abn = pd.read_csv(find_one("abnativ_summary_all_models.csv", quiet=True))
print("[출처] my_run 에 직접 만든 결과가 있으면 그것, 없으면 data/ 레퍼런스")

for gen in ["AbNatiV1", "AbNatiV2"]:
    sub = abn[abn.model_generation == gen]
    print(f"\n=== {gen} ===")
    display(sub[["abnativ_model", "variant", "overall_score", "FR_score",
                 "CDR1_score", "CDR2_score", "CDR3_score"]].round(4))

par_vh = abn[(abn.model_generation == "AbNatiV1") & (abn.variant == "parental_VH")].iloc[0]
sap_vh = abn[(abn.model_generation == "AbNatiV1") & (abn.variant == "sapiens_humanized_VH")].iloc[0]
print(f"\nAbNatiV1 VH  parental {par_vh.overall_score:.4f} → Sapiens {sap_vh.overall_score:.4f}")
print(f"  FR      {par_vh.FR_score:.4f} → {sap_vh.FR_score:.4f}   (framework 가 크게 사람 레퍼토리에 붙음)")
print(f"  CDR-H3  {par_vh.CDR3_score:.4f} → {sap_vh.CDR3_score:.4f}   (거의 불변 — CDR-H3 를 안 건드렸으니 당연)")
_v2 = abn[(abn.model_generation == "AbNatiV2") & (abn.variant == "sapiens_humanized_VH")]
print("\nAbNatiV2 VH2 humanized 실측:",
      round(float(_v2.overall_score.iloc[0]), 4),   # .iloc[0] — pandas 2.x 는 float(Series) 를 거부합니다
      "(본문의 0.6900 은 재현되지 않습니다)")

In [ ]:
from humanization_viz import nativeness_panel

sub = abn[(abn.model_generation == "AbNatiV1") & (abn.variant.str.endswith("_VH"))]
rows = [{"label": r.variant.replace("_VH", "").replace("_humanized", "").replace("_humanised", "").replace("_bestscore", ""),
         "overall": r.overall_score, "fr": r.FR_score, "cdr": r.CDR3_score}
        for r in sub.itertuples()]
nativeness_panel(rows, "AbNatiV1 VH nativeness — overall / FR / CDR-H3", "07_nativeness.png")
from IPython.display import Image; Image("07_nativeness.png")

## 5) 세 축을 한 표에 — humanness · nativeness · naturalness

세 지표는 **서로 다른 것**을 잽니다. 한 줄로 놓고 보면 어긋나는 지점이 바로 보입니다.

In [ ]:
abn1 = abn[(abn.model_generation == "AbNatiV1") & (abn.variant.str.endswith("_VH"))]
abn1_vh = pd.Series({r.variant.split("_")[0]: float(r.overall_score) for r in abn1.itertuples()})
abr_p = abr[abr.chain == "paired"].set_index("variant")["mean_logp"]

panel = pd.DataFrame({
    "AbNatiV1 VH (nativeness↑)": abn1_vh.round(4),
    "Ab-RoBERTa paired (naturalness↑)": abr_p.round(4),
}).dropna(how="all")
display(panel)
print("읽는 법 — nativeness 는 Sapiens 후보가 가장 높지만(0.8803), naturalness 로 보면 순위가 달라집니다.")
print("권장 역할 분담: 주 패널 = Sapiens humanness + Humatch CNN + AbNatiV, "
      "Ab-RoBERTa = naturalness 이상치 탐지 보조.")

## 이 랩에서 확인한 것

1. **AbNatiV 는 두 세대**입니다. 본문 7.1.3 표는 **AbNatiV1** — VH **0.6477 → 0.8803**, FR **0.6317 → 0.9245**, CDR-H3 **0.6137 → 0.6265**(거의 불변), parental VL **0.9022**. 7개 값 모두 재현됩니다. 본문의 "VL 0.902" 는 **parental** 값입니다(humanized VL 은 0.9980).
2. **AbNatiV2** 는 절반만 재현됩니다 — parental VH2 **0.4927**(일치), humanized VH2 **0.7777**(본문 0.6900 과 불일치).
3. **Ab-RoBERTa** — parental **−0.7240** · Humatch **−0.7717** 는 본문과 정확히 일치하지만, Sapiens **−0.4973**(본문 −0.6928)·AnthroAb **−0.5285**(본문 −0.8733)는 **재현되지 않습니다.** 실측값을 씁니다.
4. **naturalness ≠ humanness** — VH 만 보면 parental(−0.5188)이 가장 자연스럽습니다.

다음 → **[Ch.08 — 구조 검증](../08_structure/08_structure_lab.ipynb)**